<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de Lenguaje Natural
# Pre-training GPT-2

En este código entrenamos desde cero un modelo de lenguaje GPT2 usando el dataset Tiny Shakespeare, pasando por todo el pipeline: tokenización, creación de batches, configuración del modelo y entrenamiento. Luego utilizamos el modelo entrenado para generar texto nuevo a partir de un prompt, controlando la creatividad mediante técnicas de sampling.

Importamos lo necesario

In [4]:
import torch
from datasets import Dataset
# Herramientas de Hugging Face para modelos de lenguaje
from transformers import (
    AutoTokenizer,                    # Tokenizador automático (convierte texto a números)
    GPT2Config,                        # Configuración del modelo gPT
    GPT2LMHeadModel,                   # Modelo GPT para generación de texto (causal language model)
    DataCollatorForLanguageModeling,  # Prepara los datos para entrenamiento (batching + máscaras)
    TrainingArguments,                # Configuración del entrenamiento
    Trainer                          # Clase que maneja el entrenamiento automáticamente
)
from itertools import chain

Detectamos dispositivo

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando dispositivo:", device)

Usando dispositivo: cuda


Descargamos el dataset de Tiny Shakespeare

In [6]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

Convertimos el texto en líneas (dataset simple)

In [7]:
lines = text.split("\n")
dataset = Dataset.from_dict({"text": lines})

print("Ejemplo:", dataset[0])

Ejemplo: {'text': 'First Citizen:'}


Tokenizamos (reutilizamos GPT2)

In [10]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2 no tiene pad_token por defecto

In [11]:
tokenizer.pad_token = tokenizer.eos_token

Definimos una función para realizar tokenización con chunking eficiente

In [12]:
def chunkify(examples):
    """
    Convierte texto a tokens, concatena y divide en chunks fijos
    """

    # Tokenizamos
    tokens = tokenizer(examples["text"])

    # Aplanamos la lista de listas
    concat = list(chain(*tokens["input_ids"]))

    chunk_size = 256

    # Creamos bloques de tamaño fijo
    chunks = [
        concat[i:i+chunk_size]
        for i in range(0, len(concat) - chunk_size + 1, chunk_size)
    ]

    # Attention mask (todo 1 porque no hay padding interno)
    attention_mask = [[1] * chunk_size for _ in chunks]

    return {
        "input_ids": chunks,
        "attention_mask": attention_mask
    }

Aplicamos transformación

In [13]:
tokenized = dataset.map(
    chunkify,
    batched=True,
    remove_columns=["text"]
)


print("Ejemplo tokenizado:", tokenized[0])

Map:   0%|          | 0/40001 [00:00<?, ? examples/s]

Ejemplo tokenizado: {'input_ids': [5962, 22307, 25, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 3237, 25, 5248, 461, 11, 2740, 13, 5962, 22307, 25, 1639, 389, 477, 12939, 2138, 284, 4656, 621, 284, 1145, 680, 30, 3237, 25, 4965, 5634, 13, 12939, 13, 5962, 22307, 25, 5962, 11, 345, 760, 327, 1872, 385, 1526, 28599, 318, 4039, 4472, 284, 262, 661, 13, 3237, 25, 1135, 760, 470, 11, 356, 760, 470, 13, 5962, 22307, 25, 5756, 514, 1494, 683, 11, 290, 356, 1183, 423, 11676, 379, 674, 898, 2756, 13, 3792, 470, 257, 15593, 30, 3237, 25, 2949, 517, 3375, 319, 470, 26, 1309, 340, 307, 1760, 25, 1497, 11, 1497, 0, 12211, 22307, 25, 3198, 1573, 11, 922, 4290, 13, 5962, 22307, 25, 1135, 389, 17830, 3595, 4290, 11, 262, 1458, 1173, 1547, 922, 13, 2061, 4934, 969, 5036, 896, 319, 561, 26958, 514, 25, 611, 484, 19188, 7800, 514, 475, 262, 48713, 414, 11, 981, 340, 547, 1929, 4316, 462, 11, 356, 1244, 4724, 484, 22598, 514, 31533, 306, 26, 4360, 484, 892, 356, 389, 1165, 13674, 25, 262, 10904, 

Realizamos el split de train y validation

In [14]:
split = tokenized.train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

Definimos modelo GPT2 (desde cero)

In [15]:
config = GPT2Config(
    vocab_size=len(tokenizer),

    # Longitud máxima
    max_position_embeddings=256,

    # Arquitectura (ligera para Colab)
    hidden_size=512,
    num_hidden_layers=8,
    num_attention_heads=8,
    dropout=0.1
)

model = GPT2LMHeadModel(config).to(device)

print("Parámetros del modelo:", model.num_parameters())

Parámetros del modelo: 51082752


Definimos Data Collator (Causal LM)

In [16]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # IMPORTANTE: causal, no masked LM
)

Definimos Training Arguments (optimizados)

In [17]:
training_args = TrainingArguments(
    output_dir="./tiny_gpt_pretrain",

    # Batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    # Entrenamiento
    num_train_epochs=5,   # número de veces que el modelo ve todo el dataset
    learning_rate=5e-4,   # CLAVE para pretraining desde cero
    warmup_steps=200,

    # Regularización
    weight_decay=0.01, # penaliza pesos grandes → modelo más generalizable

    # Logging y evaluación
    logging_steps=50, # cada cuántos pasos imprime métricas (loss, etc.)
    save_steps=200, # cada cuántos pasos guarda el modelo

    # Performance
    fp16=True if device == "cuda" else False,

    # Otros
    report_to="none"  # Desactiva reportes a herramientas externas (wandb, etc.)
)

Definimos el trainer

In [18]:
trainer = Trainer(
    model=model,                 # El modelo que queremos entrenar (GPT2)
    args=training_args,          # Configuración del entrenamiento (learning rate, epochs, batch, etc.)
    train_dataset=train_dataset, # Dataset de entrenamiento (datos que el modelo usa para aprender)
    eval_dataset=eval_dataset,   # Dataset de evaluación (para medir qué tan bien está aprendiendo)
    data_collator=data_collator  # Función que prepara los batches (padding, labels, formato correcto)
)

Entrenamos

In [19]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,9.618166
100,7.137883
150,6.561754
200,6.130886
250,5.748519
300,5.371578
350,5.283432
400,5.114139
450,4.919809
500,4.889243


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=645, training_loss=5.765005090994428, metrics={'train_runtime': 106.9167, 'train_samples_per_second': 48.122, 'train_steps_per_second': 6.033, 'total_flos': 199307357061120.0, 'train_loss': 5.765005090994428, 'epoch': 5.0})

Generamos texto

In [20]:
model.eval()

prompt = "ROMEO:"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_length=200,

    # Sampling (mejor calidad)
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,

    pad_token_id=tokenizer.eos_token_id
)

print("\n=== GENERACIÓN ===\n")
print(tokenizer.decode(output[0], skip_special_tokens=True))


=== GENERACIÓN ===

ROMEO:Is it is my heart in thyself and good is not no's daughter;I am a thousand day,And yet I can I'll make her well.ROMEO:The old daughter;But, I pray it is he is a lady.ROMEO:He shall be so.MERCUTIO:I think it not; for that she do say, my good lady and I had seen his father, let's name of love a gentleman!BENVOLIO:Why, in a good, that'st.ROMEO:A most more?MERCUTIO:You have it, goodca, or thou art thou art thou hast a thousand-night, good father, she's, by the dost:How was a shrewd?MERCUTIO:She hath he will be the prince! O, sir.ROMEO: I'll be a good to thee in her son.ROMEO:She is a good father's a good
